In [ ]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
from rapidfuzz import fuzz

In [ ]:
MODEL_PATH        = "/mnt/storage_C1/igorzwirtes/poster_ic/qwen2.5coder"
ADAPTER_PATH      = "/mnt/storage_C1/igorzwirtes/poster_ic/lora_weights/B"
SPIDER_TABLES     = "/mnt/storage_C1/igorzwirtes/poster_ic/spider/tables.json"
FILE_PATH         = "/mnt/storage_C1/igorzwirtes/poster_ic/predictions/B2.json"

USE_BF16          = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16          = torch.cuda.is_available() and not USE_BF16
USE_CPU           = not torch.cuda.is_available()
MAX_PROMPT_TOKENS = 2048    
TARGET_MAX        = 1024   

In [ ]:
def get_relevant_tables(question: str, db: dict, top_k: int = 8) -> set[int]:
    q = question.lower()
    scores = {}

    for i, table in enumerate(db["table_names_original"]):
        table_score = fuzz.partial_ratio(table.lower(), q)

        cols = [c[1].lower() for c in db["column_names_original"] if c[0] == i]
        col_score = max((fuzz.partial_ratio(c, q) for c in cols), default=0)

        scores[i] = max(table_score, col_score * 0.8)

    # ranking + filtro leve
    top = sorted(scores, key=scores.get, reverse=True)
    top = [i for i in top if scores[i] > 5][:top_k]

    seed = set(top)

    # FK expansion (1-hop)
    fk_tables = set(seed)

    for fk in db.get("foreign_keys", []):
        t1 = db["column_names_original"][fk[0]][0]
        t2 = db["column_names_original"][fk[1]][0]

        if t1 in seed or t2 in seed:
            fk_tables.add(t1)
            fk_tables.add(t2)

    return fk_tables

In [ ]:
with open(SPIDER_TABLES) as f:
    tables_data = json.load(f)

def format_schema_compact(db: dict, question: str, max_cols: int = 8) -> str:
    col_names = db["column_names_original"]
    col_types = db["column_types"]
    pks       = set(db.get("primary_keys", []))

    fk_map = {}
    for fk in db.get("foreign_keys", []):
        src, ref      = fk[0], fk[1]
        ref_table     = db["table_names_original"][col_names[ref][0]]
        ref_col       = col_names[ref][1]
        fk_map.setdefault(src, []).append(f"{ref_table}.{ref_col}")

    relevant_tables = get_relevant_tables(question, db)
    q_tokens        = set(re.sub(r"[^\w\s]", "", question.lower()).split())

    lines = []
    for i, table in enumerate(db["table_names_original"]):
        if i not in relevant_tables:
            continue

        cols_data = [
            (idx, col[1], col_types[idx])
            for idx, col in enumerate(col_names)
            if col[0] == i
        ]

        def col_score(item):
            idx, name, _ = item
            score = fuzz.partial_ratio(name.lower(), question.lower())
            if any(t in name.lower() for t in q_tokens): score += 20
            if idx in pks:    score += 30
            if idx in fk_map: score += 25
            return score

        cols_sorted = sorted(cols_data, key=col_score, reverse=True)[:max_cols]
        cols_sorted.sort(key=lambda x: x[0])

        col_parts = []
        for idx, name, ctype in cols_sorted:
            part = f"{name} {ctype}"
            if idx in pks:    part += " PK"
            if idx in fk_map: part += f" FK→{fk_map[idx]}"
            col_parts.append(part)

        lines.append(f"{table}({', '.join(col_parts)})")

    return "\n".join(lines)

tables_index = {db["db_id"]: db for db in tables_data}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

In [ ]:
dataset    = load_dataset("xlangai/spider")
test_data   = dataset["validation"]

def build_prompt(example: dict) -> str:
    db = tables_index[example["db_id"]]
    q  = example["question"]

    for max_cols in [8, 6, 4, 3]:
        schema = format_schema_compact(db, q, max_cols=max_cols)

        messages = [
            {"role": "system", "content": "Convert the question to a valid SQLite query. Output SQL only. Always use table aliases when joining multiple tables."},
            {"role": "user",   "content": f"Schema ({example['db_id']}):\n{schema}\n\nQuestion: {q}"},
        ]

        prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        n_tokens = len(tokenizer.encode(prompt))

        if n_tokens <= TARGET_MAX:
            break

    # hard cap: trunca linhas do schema até caber
    if n_tokens > MAX_PROMPT_TOKENS:
        schema_lines = schema.split("\n")
        while len(schema_lines) > 1 and n_tokens > MAX_PROMPT_TOKENS:
            schema_lines.pop()
            messages[1]["content"] = f"Schema ({example['db_id']}):\n{chr(10).join(schema_lines)}\n\nQuestion: {q}"
            prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            n_tokens = len(tokenizer.encode(prompt))

    return prompt

# Testar uso máximo de VRAM
#train_data = sorted(train_data, key=lambda ex: len(tokenizer(build_prompt(ex))["input_ids"]), reverse=True)

In [ ]:
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.config.use_cache = True

model.eval();

In [ ]:
def extract_sql(text):
    # bloco markdown
    match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        text,
        re.DOTALL | re.IGNORECASE
    )

    if match:
        sql = match.group(1).strip()
    else:
        # pega primeira query SQL
        match = re.search(
            r"(SELECT|INSERT|UPDATE|DELETE|WITH)\b.*?;",
            text,
            re.DOTALL | re.IGNORECASE
        )

        if match:
            sql = match.group(0).strip()
        else:
            sql = text.strip()

    # remove comentários
    sql = re.sub(r"--.*", "", sql)

    # remove markdown sobrando
    sql = sql.replace("```", "").strip()

    return sql

In [ ]:
predictions = []

for example in test_data:
    prompt = build_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            early_stopping=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<|im_end|>"),
            ],
        )

    # Decodifica só os tokens novos
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = extract_sql(
        tokenizer.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
    )
    pred_sql = pred_sql.split(";")[0] + ";"

    predictions.append({
        "db_id": example["db_id"],
        "question": example["question"],
        "gold": example["query"],
        "predicted": pred_sql,
    })
    print(f"Q: {example['question']}\nGold: {example['query']}\nPred: {pred_sql}\n---")

with open(FILE_PATH, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"\nPredições salvas: {len(predictions)} exemplos")